# Vérifier la structure des données

In [79]:
from pathlib import Path
import json
from rich.tree import Tree
from rich import print as rprint

## Chargement données

In [80]:
def recuperer_recent(chemin_dossier)  -> list :
    '''
    Récupérer le chemin du document json le plus récent.
    Entrée : chemin du dossier visé
    Sortie : chemin du fichier horodaté le plus récent
    '''
    fichiers = []
    if chemin_dossier.exists():
        fichiers_ft = sorted(chemin_dossier.glob("*.json"))
        if fichiers_ft:
            fichiers.append(str(fichiers_ft[-1]))  # le plus récent  
    return fichiers[0]
#====================================================================================s
def charger_offres(chemin_fichier):
    '''
    Charger les offres du chemin du fichier json renseigné en paramètre
    '''
    try:
        with open(chemin_fichier, "r", encoding="utf-8") as f:
            offres = json.load(f)
        print(f"  {chemin_fichier} → {len(offres)} offres")
        return offres
    except Exception as e:
        print(f"  Erreur chargement {chemin_fichier} : {e}")


#====================================================================================
def json_vers_arbre(data, arbre, prefixe=""):
    """
    Construit récursivement un arbre Rich
    à partir d'un dictionnaire JSON.
    """
    if isinstance(data, dict):
        for cle, valeur in data.items():
            if isinstance(valeur, dict):
                # Sous-dictionnaire → nouveau nœud
                noeud = arbre.add(f"[bold cyan]{cle}[/] 📁")
                json_vers_arbre(valeur, noeud)

            elif isinstance(valeur, list):
                # Tableau → nouveau nœud avec indication du type
                noeud = arbre.add(f"[bold yellow]{cle}[/] 📋 [dim]({len(valeur)} éléments)[/]")
                if valeur and isinstance(valeur[0], dict):
                    json_vers_arbre(valeur[0], noeud)

            else:
                # Valeur simple → feuille
                type_valeur = type(valeur).__name__
                arbre.add(f"[green]{cle}[/] [dim]({type_valeur})[/]")

#====================================================================================
def afficher_structure_json(data, titre="Structure JSON"):
    """
    Affiche la structure d'un JSON sous forme d'arbre visuel.
    """
    arbre = Tree(f"[bold magenta]{titre}[/]")
    json_vers_arbre(data, arbre)
    rprint(arbre)

In [81]:
# ── Étape 1 : Récupération du nom du fichier le plus récent ──────────────────────────────
chemin_fichier_raw_ft = recuperer_recent(Path("../data/raw/francetravail"))
chemin_fichier_raw_wttj = recuperer_recent(Path("../data/raw/welcometothejungle"))

chemin_fichier_processed_ft = recuperer_recent(Path("../data/processed/francetravail"))
chemin_fichier_processed_wttj = recuperer_recent(Path("../data/processed/welcometothejungle"))

chemin_fichier_normalise = recuperer_recent(Path("../data/processed/normalise"))

# ── Étape 2 : Récupération des offres dans Python (liste de dictionnaires) ──────────────
# Offres brutes issues de l'API France Travail et du scraping du site Welcome To The Jungle
offres_brutes_ft =  charger_offres(chemin_fichier_raw_ft)
offres_brutes_wttj =  charger_offres(chemin_fichier_raw_wttj)

# Offres parsees - traitement minimal (suppression des balises html, etc... 
offres_parsees_ft =  charger_offres(chemin_fichier_processed_ft)
offres_parsees_wttj =  charger_offres(chemin_fichier_processed_wttj)

# Offres normalisées
offres_normalisees =  charger_offres(chemin_fichier_normalise)

  ../data/raw/francetravail/data_engineer_20260407_142250.json → 578 offres
  ../data/raw/welcometothejungle/offres_20260407_142255.json → 1000 offres
  ../data/processed/francetravail/offres_20260407_142250.json → 578 offres
  ../data/processed/welcometothejungle/offres_20260407_142255.json → 1000 offres
  ../data/processed/normalise/offres_20260407_142335.json → 1578 offres


## Affichage structure

### Données brutes - France Travail

In [82]:
# Afficher 
titre = "structure données brutes - France Travail"
sample_file = offres_brutes_ft[0]

afficher_structure_json(sample_file, titre)
rprint(sample_file)

structure données brutes - France Travail
├── id (str)
├── intitule (str)
├── description (str)
├── dateCreation (str)
├── dateActualisation (str)
├── lieuTravail 📁
│   └── libelle (str)
├── romeCode (str)
├── romeLibelle (str)
├── appellationlibelle (str)
├── entreprise 📁
│   ├── nom (str)
│   └── entrepriseAdaptee (bool)
├── typeContrat (str)
├── typeContratLibelle (str)
├── natureContrat (str)
├── experienceExige (str)
├── experienceLibelle (str)
├── langues 📋 (1 éléments)
│   ├── libelle (str)
│   └── exigence (str)
├── competences 📋 (1 éléments)
│   ├── code (str)
│   ├── libelle (str)
│   └── exigence (str)
├── salaire 📁
│   └── libelle (str)
├── dureeTravailLibelle (str)
├── dureeTravailLibelleConverti (str)
├── alternance (bool)
├── contact 📁
│   ├── nom (str)
│   ├── coordonnees1 (str)
│   └── courriel (str)
├── agence 📁
├── nombrePostes (int)
├── accessibleTH (bool)
├── deplacementCode (str)
├── deplacementLibelle (str)
├── qualificationCode (str)
├── qualificationLibelle (str)
├── codeNAF (str)
├── secteurActivite (str)
├── secteurActiviteLibelle (str)
├── trancheEffectifEtab (str)
├── origineOffre 📁
│   ├── origine (str)
│   └── urlOrigine (str)
├── offresManqueCandidats (bool)
├── contexteTravail 📁
│   └── horaires 📋 (1 éléments)
├── entrepriseAdaptee (bool)
└── employeurHandiEngage (bool)

{
    'id': '206MWBD',
    'intitule': 'Data Engineer IA Copilot Studio & Automatisation IA (H/F)',
    'description': 'Votre rôle se décline en 4 missions.\n\n1. Architecte d\'Agents IA (Copilot Studio & RAG) : 
\n\n- Conception d\'agents conversationnels évolués : Développer des copilotes capables de comprendre des contextes
métiers complexes (réservations, logistique, support client).\n- Maîtrise du RAG (Retrieval-Augmented Generation) :
Connecter les agents aux bases de connaissances internes via Azure AI Search pour garantir des réponses précises, 
sourcées et sécurisées.\n- Optimisation LLM : Intégrer et paramétrer les modèles via Azure OpenAI (GPT-4o, etc.) en
mettant l\'accent sur le Prompt Engineering de haut niveau.\n\n2. Orchestration & Automatisation "Agentique" : 
\n\n- Workflows Actionnables : Utiliser Power Automate pour permettre aux copilotes de déclencher des actions 
réelles (mise à jour de CRM, envoi de rapports, validation de workflows).\n- Interconnexion : Développer des 
connecteurs personnalisés pour lier la Power Platform aux APIs propriétaires du groupe.\n\n3. Ingénierie de Données
pour l\'IA : \n\n- Pipelines Data : Industrialiser les flux d\'ingestion (Batch/Real-time) via Azure Data Factory 
pour alimenter les moteurs d\'indexation.\n- Gouvernance & Qualité : Structurer les données non structurées (PDF, 
wikis, bases documentaires) pour maximiser la pertinence des systèmes RAG.\n\n4. Innovation et acculturation : 
\n\n- Cadrage métier : Animer des ateliers avec les directions opérationnelles pour identifier les cas 
d\'utilisation à ROI immédiat.\n- MVP & Scale : Passer rapidement du prototype à une solution robuste, surveillée 
et sécurisée (AI Guardrails).',
    'dateCreation': '2026-04-07T10:09:08.853Z',
    'dateActualisation': '2026-04-07T10:22:31.699Z',
    'lieuTravail': {'libelle': '75 - Paris (Dept.)'},
    'romeCode': 'M1811',
    'romeLibelle': 'Data engineer',
    'appellationlibelle': 'Data engineer',
    'entreprise': {'nom': 'BASSETTI GROUP', 'entrepriseAdaptee': False},
    'typeContrat': 'CDI',
    'typeContratLibelle': 'CDI',
    'natureContrat': 'Contrat travail',
    'experienceExige': 'E',
    'experienceLibelle': '3 An(s)',
    'langues': [{'libelle': 'Anglais', 'exigence': 'S'}],
    'competences': [
        {'code': '300067', 'libelle': 'Analyser, exploiter, structurer des données', 'exigence': 'S'}
    ],
    'salaire': {'libelle': 'Annuel de 45000.0 Euros à 65000.0 Euros sur 12.0 mois'},
    'dureeTravailLibelle': '35H/semaine\nTravail en journée',
    'dureeTravailLibelleConverti': 'Temps plein',
    'alternance': False,
    'contact': {
        'nom': 'BASSETTI GROUP - M. Benjamin COULOMB',
        'coordonnees1': 'Pour postuler, utiliser le lien suivant : 
https://candidat.francetravail.fr/offres/recherche/detail/206MWBD',
        'courriel': 'Pour postuler, utiliser le lien suivant : 
https://candidat.francetravail.fr/offres/recherche/detail/206MWBD'
    },
    'agence': {},
    'nombrePostes': 1,
    'accessibleTH': False,
    'deplacementCode': '1',
    'deplacementLibelle': 'Jamais',
    'qualificationCode': '6',
    'qualificationLibelle': 'Employé qualifié',
    'codeNAF': '64.20Z',
    'secteurActivite': '64',
    'secteurActiviteLibelle': 'Activités des sociétés holding',
    'trancheEffectifEtab': '3 à 5 salariés',
    'origineOffre': {
        'origine': '1',
        'urlOrigine': 'https://candidat.francetravail.fr/offres/recherche/detail/206MWBD'
    },
    'offresManqueCandidats': False,
    'contexteTravail': {'horaires': ['35H/semaine\nTravail en journée']},
    'entrepriseAdaptee': False,
    'employeurHandiEngage': False
}

### Données brutes - Welcome to the Jungle

In [83]:
# Afficher 
titre = "structure données brutes - Welcome to the Jungle"
sample_file = offres_brutes_wttj[0]

afficher_structure_json(sample_file, titre)
rprint(sample_file)

structure données brutes - Welcome to the Jungle
├── has_education_level (bool)
├── contract_duration_maximum (NoneType)
├── published_at_date (str)
├── reference (str)
├── has_salary_yearly_minimum (bool)
├── salary_currency (str)
├── new_profession 📁
│   ├── sub_category_reference (str)
│   ├── sub_category_name (str)
│   ├── category_reference (str)
│   ├── category_name (str)
│   ├── pivot_name (str)
│   └── pivot_reference (str)
├── salary_minimum (int)
├── is_boosted (bool)
├── benefits 📋 (12 éléments)
├── source_stage (str)
├── has_remote (bool)
├── contract_type (str)
├── slug (str)
├── has_experience_level_minimum (bool)
├── summary (str)
├── rank_group_1 (int)
├── organization_score (int)
├── published_at_timestamp (int)
├── offices 📋 (1 éléments)
│   ├── state (str)
│   ├── city (str)
│   ├── country (str)
│   ├── country_code (str)
│   ├── district (str)
│   ├── local_district (str)
│   ├── local_city (str)
│   └── local_state (str)
├── language (str)
├── profile_ranking (int)
├── rank_group_2 (int)
├── _geoloc 📋 (1 éléments)
│   ├── lat (float)
│   └── lng (float)
├── remote (str)
├── experience_level_minimum (int)
├── organization 📁
│   ├── name (str)
│   ├── description (NoneType)
│   ├── reference (str)
│   ├── labels 📋 (0 éléments)
│   ├── summary (str)
│   ├── profile_type (str)
│   ├── slug (str)
│   ├── logo 📁
│   │   ├── url (str)
│   │   └── thumb 📁
│   │       └── url (str)
│   ├── nb_employees (int)
│   ├── creation_year (int)
│   ├── cover_image 📁
│   │   ├── small 📁
│   │   │   └── url (str)
│   │   ├── url (str)
│   │   ├── medium 📁
│   │   │   └── url (str)
│   │   ├── large 📁
│   │   │   └── url (str)
│   │   └── social 📁
│   │       └── url (str)
│   ├── profile_ranking (int)
│   ├── commitments 📋 (0 éléments)
│   └── equality_index (int)
├── has_benefits (bool)
├── salary_period (str)
├── rank_group_3 (int)
├── wk_reference (str)
├── key_missions 📋 (3 éléments)
├── education_level (str)
├── contract_duration_minimum (NoneType)
├── profile (str)
├── name (str)
├── has_contract_duration (bool)
├── published_at (str)
├── salary_yearly_minimum (int)
├── salary_maximum (int)
├── sectors 📋 (2 éléments)
│   ├── name (str)
│   ├── reference (str)
│   ├── parent_name (str)
│   └── parent_reference (str)
├── objectID (str)
└── _highlightResult 📁
    ├── new_profession 📁
    │   ├── sub_category_reference 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   ├── fullyHighlighted (bool)
    │   │   └── matchedWords 📋 (1 éléments)
    │   ├── sub_category_name 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   ├── category_name 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   ├── pivot_name 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   └── pivot_reference 📁
    │       ├── value (str)
    │       ├── matchLevel (str)
    │       ├── fullyHighlighted (bool)
    │       └── matchedWords 📋 (2 éléments)
    ├── summary 📁
    │   ├── value (str)
    │   ├── matchLevel (str)
    │   ├── fullyHighlighted (bool)
    │   └── matchedWords 📋 (2 éléments)
    ├── offices 📋 (1 éléments)
    │   ├── state 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   ├── city 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   ├── country 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   ├── country_code 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   ├── district 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   ├── local_district 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │  

{
    'has_education_level': True,
    'contract_duration_maximum': None,
    'published_at_date': '2026-04-07',
    'reference': 'edfc923b-f45a-42ba-8663-718f40dda670',
    'has_salary_yearly_minimum': True,
    'salary_currency': 'EUR',
    'new_profession': {
        'sub_category_reference': 'data-business-intelligence-yZjY1',
        'sub_category_name': 'Données/Business Intelligence',
        'category_reference': 'tech-engineering-3NjUy',
        'category_name': 'Technologie et ingénierie',
        'pivot_name': 'Ingénieur de données',
        'pivot_reference': 'data-engineer-wOTA3'
    },
    'salary_minimum': 43000,
    'is_boosted': False,
    'benefits': [
        'Entre 1-2 jours de télétravail',
        'Animaux acceptés',
        'Congés payés supplémentaires',
        'Prime de cooptation',
        'Team building',
        "Afterworks, Déjeuners d'équipe, etc.",
        'Aide à la garde d’enfant, Crèche...',
        'Congés pour enfant malade',
        'Salle de sport dans les locaux',
        'Parking à vélo',
        'Remboursement au delà des 50%',
        'Parking'
    ],
    'source_stage': 'production',
    'has_remote': True,
    'contract_type': 'full_time',
    'slug': 'analytics-engineer-h-f_paris_VENTE_NQMqm57',
    'has_experience_level_minimum': True,
    'summary': "Rejoins Vente-unique.com, une entreprise dynamique et en pleine croissance dans le secteur du 
e-commerce. En tant qu'Analytics Engineer, tu seras au cœur de l'ingénierie et de l'analyse de données, travaillant
en étroite collaboration avec les équipes métiers pour transformer leurs besoins en solutions concrètes. Tu auras 
l'opportunité de contribuer à des projets à fort impact business et de diffuser la culture data au sein de 
l'entreprise.",
    'rank_group_1': 6850,
    'organization_score': 80,
    'published_at_timestamp': 1775554905,
    'offices': [
        {
            'state': 'Ile-de-France',
            'city': 'Paris',
            'country': 'France',
            'country_code': 'FR',
            'district': 'Paris',
            'local_district': 'Paris',
            'local_city': 'Paris',
            'local_state': 'Île-de-France'
        }
    ],
    'language': 'fr',
    'profile_ranking': 100,
    'rank_group_2': 6849,
    '_geoloc': [{'lat': 48.88243, 'lng': 2.38244}],
    'remote': 'punctual',
    'experience_level_minimum': 1,
    'organization': {
        'name': 'Vente-unique.com',
        'description': None,
        'reference': 'KADDeeJ',
        'labels': [],
        'summary': "Mobilier et décoration en ligne pour 14 pays d'Europe.",
        'profile_type': 'standard',
        'slug': 'vente-unique',
        'logo': {
            'url': 
'https://cdn-images.welcometothejungle.com/Ozf9otYyAdpQ7uxOby8t56SUEabd1L7Nb0DNAiBTaWQ/rs:auto:400::/q:85/czM6Ly93d
HRqLXByb2R1Y3Rpb24vdXBsb2Fkcy9vcmdhbml6YXRpb24vbG9nby84MTI1LzE1NTk1Ny8yNWQ4N2ZmNC04NTY0LTQ4M2UtYTBkMy0yMDZkMTkyMDM0
YTguanBn',
            'thumb': {
                'url': 
'https://cdn-images.welcometothejungle.com/WJBUzTxl6av6mSKApskQGPeu8Fe3LmIum4UOecBEQJU/rs:auto:70::/q:85/czM6Ly93dH
RqLXByb2R1Y3Rpb24vdXBsb2Fkcy9vcmdhbml6YXRpb24vbG9nby84MTI1LzE1NTk1Ny8yNWQ4N2ZmNC04NTY0LTQ4M2UtYTBkMy0yMDZkMTkyMDM0Y
TguanBn'
            }
        },
        'nb_employees': 240,
        'creation_year': 2006,
        'cover_image': {
            'small': {
                'url': 
'https://cdn-images.welcometothejungle.com/IjTWbT7aWaqqCsSUqPCtnzNrYWGBHVuGGgCIJX1o1BI/rs:auto:640::/q:85/czM6Ly93d
HRqLXByb2R1Y3Rpb24vdXBsb2Fkcy93ZWJzaXRlX29yZ2FuaXphdGlvbi9jb3Zlcl9pbWFnZS93dHRqX2ZyL2ZyLWMwMDEyYzI4LWEwYmYtNDJlMy04
OTVhLTkyYWFhNTQ3ODgyMy5qcGc'
            },
            'url': 
'https://cdn-images.welcometothejungle.com/90P5IyB2EhcrnDxxlhvYfZs1LO-qKOr1PSJ_whJa6is/rs:auto:2000:450:/g:fp:0:0.5
/q:85/czM6Ly93dHRqLXByb2R1Y3Rpb24vdXBsb2Fkcy93ZWJzaXRlX29yZ2FuaXphdGlvbi9jb3Zlcl9pbWFnZS93dHRqX2ZyL2ZyLWMwMDEyYzI4L
WEwYmYtNDJlMy04OTVhLTkyYWFhNTQ3ODgyMy5qcGc',
            'medium': {
       

### Données parsees - France Travail

In [84]:
# Afficher 
titre = "structure données parsées - France Travail"
sample_file = offres_parsees_ft[0]

afficher_structure_json(sample_file, titre)
rprint(sample_file)

structure données parsées - France Travail
├── id (str)
├── url (str)
├── titre (str)
├── description (str)
├── entreprise (str)
├── nb_employes (str)
├── type_contrat (str)
├── type_contrat_libelle (str)
├── alternance (bool)
├── nombre_postes (int)
├── temps_travail (str)
├── salaire_min (int)
├── salaire_max (int)
├── salaire_texte (str)
├── experience_exige (str)
├── experience_min (int)
├── qualification (str)
├── localisation_ville (str)
├── localisation_dept (NoneType)
├── commune (NoneType)
├── latitude (NoneType)
├── longitude (NoneType)
├── teletravail (NoneType)
├── secteur (str)
├── code_naf (str)
├── rome_code (str)
├── rome_libelle (str)
├── competences 📋 (1 éléments)
├── langues 📋 (1 éléments)
├── qualites 📋 (0 éléments)
├── url_postulation (NoneType)
├── date_publication (str)
├── date_actualisation (str)
├── date_extraction (str)
└── source (str)

{
    'id': '206MWBD',
    'url': 'https://candidat.francetravail.fr/offres/recherche/detail/206MWBD',
    'titre': 'Data engineer',
    'description': 'Votre rôle se décline en 4 missions.\n\n1. Architecte d\'Agents IA (Copilot Studio & RAG) : 
\n\n- Conception d\'agents conversationnels évolués : Développer des copilotes capables de comprendre des contextes
métiers complexes (réservations, logistique, support client).\n- Maîtrise du RAG (Retrieval-Augmented Generation) :
Connecter les agents aux bases de connaissances internes via Azure AI Search pour garantir des réponses précises, 
sourcées et sécurisées.\n- Optimisation LLM : Intégrer et paramétrer les modèles via Azure OpenAI (GPT-4o, etc.) en
mettant l\'accent sur le Prompt Engineering de haut niveau.\n\n2. Orchestration & Automatisation "Agentique" : 
\n\n- Workflows Actionnables : Utiliser Power Automate pour permettre aux copilotes de déclencher des actions 
réelles (mise à jour de CRM, envoi de rapports, validation de workflows).\n- Interconnexion : Développer des 
connecteurs personnalisés pour lier la Power Platform aux APIs propriétaires du groupe.\n\n3. Ingénierie de Données
pour l\'IA : \n\n- Pipelines Data : Industrialiser les flux d\'ingestion (Batch/Real-time) via Azure Data Factory 
pour alimenter les moteurs d\'indexation.\n- Gouvernance & Qualité : Structurer les données non structurées (PDF, 
wikis, bases documentaires) pour maximiser la pertinence des systèmes RAG.\n\n4. Innovation et acculturation : 
\n\n- Cadrage métier : Animer des ateliers avec les directions opérationnelles pour identifier les cas 
d\'utilisation à ROI immédiat.\n- MVP & Scale : Passer rapidement du prototype à une solution robuste, surveillée 
et sécurisée (AI Guardrails).',
    'entreprise': 'BASSETTI GROUP',
    'nb_employes': '3 à 5 salariés',
    'type_contrat': 'CDI',
    'type_contrat_libelle': 'CDI',
    'alternance': False,
    'nombre_postes': 1,
    'temps_travail': 'Temps plein',
    'salaire_min': 45000,
    'salaire_max': 65000,
    'salaire_texte': 'Annuel de 45000.0 Euros à 65000.0 Euros sur 12.0 mois',
    'experience_exige': 'E',
    'experience_min': 3,
    'qualification': 'Employé qualifié',
    'localisation_ville': 'Paris (Dept.)',
    'localisation_dept': None,
    'commune': None,
    'latitude': None,
    'longitude': None,
    'teletravail': None,
    'secteur': 'Activités des sociétés holding',
    'code_naf': '64.20Z',
    'rome_code': 'M1811',
    'rome_libelle': 'Data engineer',
    'competences': ['Analyser, exploiter, structurer des données'],
    'langues': ['Anglais'],
    'qualites': [],
    'url_postulation': None,
    'date_publication': '2026-04-07',
    'date_actualisation': '2026-04-07',
    'date_extraction': '2026-04-07T14:22:55.050997',
    'source': 'francetravail'
}

### Données parsees - Welcome to the Jungle

In [85]:
# Afficher 
titre = "structure données parsées - Welcome to the Jungle"
sample_file = offres_parsees_wttj[0]

afficher_structure_json(sample_file, titre)
rprint(sample_file)

structure données parsées - Welcome to the Jungle
├── id (str)
├── slug (str)
├── url (str)
├── titre (str)
├── description (str)
├── missions 📋 (3 éléments)
├── type_contrat (str)
├── teletravail (str)
├── salaire_min (int)
├── salaire_max (int)
├── salaire_devise (str)
├── experience_min (int)
├── localisation_ville (str)
├── localisation_region (str)
├── pays (str)
├── latitude (float)
├── longitude (float)
├── entreprise (str)
├── entreprise_slug (str)
├── nb_employes (int)
├── entreprise_desc (str)
├── secteur (str)
├── sous_secteur (str)
├── avantages 📋 (12 éléments)
├── metier (str)
├── categorie_metier (str)
├── date_publication (str)
├── date_extraction (str)
└── source (str)

{
    'id': 'edfc923b-f45a-42ba-8663-718f40dda670',
    'slug': 'analytics-engineer-h-f_paris_VENTE_NQMqm57',
    'url': 
'https://www.welcometothejungle.com/fr/companies/vente-unique/jobs/analytics-engineer-h-f_paris_VENTE_NQMqm57',
    'titre': 'Analytics Engineer (H/F)',
    'description': 'Rejoins Vente-unique.com, une entreprise dynamique et en pleine croissance dans le secteur du 
e-commerce. En tant qu\'Analytics Engineer, tu seras au cœur de l\'ingénierie et de l\'analyse de données, 
travaillant en étroite collaboration avec les équipes métiers pour transformer leurs besoins en solutions 
concrètes. Tu auras l\'opportunité de contribuer à des projets à fort impact business et de diffuser la culture 
data au sein de l\'entreprise.\n\nProfil recherché : Diplômé.e d’une école d’ingénieur ou d’informatique, tu as 
minimum 1 an d’expérience en Data Engineer ou Analytics Engineer, avec une forte pratique de Snowflake. Nous avons 
besoin de quelqu’un qui matche avec nos valeurs : esprit d’équipe, convivialité, performance et satisfaction client
! On attend de toi également : ⭐ Hard skills : * Tu maitrises parfaitement SQL et ses différents cas 
d’applications dans Snowflake. * Tu as déjà orchestré des pipelines avec du Python, des API ou des procédures 
stockée. * Tu as déjà réalisé des modèles sémantiques complexes dans PowerBI (ou équivalent). * Tu parles 
couramment anglais et tu es à l’aise dans un environnement international. * Une première expérience sur des sujets 
de classification / Machine Learning serait un plus. ⭐Soft skills : * Orienté.e business, tu sais cadrer et 
synthétiser des besoins métiers parfois flous pour en isoler la finalité et l’impact business. * Problem solver, tu
sais décomposer des problématiques complexes pour construire la meilleure solution. * Team player, tu échanges, 
partages et co-construis pour maximiser l’impact des projets. * Curieux.se et persévérant.e, tu sais trouver les 
données nécessaires, même dans des contextes techniques exigeants. Aussi, on espère que tu as un fort attrait pour 
l’univers du e-commerce et de l’aménagement de la maison ! 🛍️ Le petit mot du Manager "🤝 Rejoins-nous si : * Tu 
apprécies un management de proximité, où ton responsable est toujours disponible pour t’accompagner et répondre à 
tes questions. * Tu sais profiter des pauses déjeuner, surtout autour d’un bon menu mix grill au resto libanais !" 
Le processus de recrutement : Il se déroule généralement en 4 étapes : 1- Préqualification téléphonique (10-15 
minutes) 2- Test de personnalité en ligne et entretien RH (motivation, parcours, challenge des résultats du test) 
3- Entretien Manager avec Eliott, ton N+1 (Test technique de 30 min en direct et 1h d’échanges) 4- Entretien final 
avec Cyrille, ton N+2, Directeur Supply Chain (fit, 30 min) Conditions & Avantages : * Type de contrat : CDI * À 
pourvoir dès que possible * Rémunération selon profil composée d’un fixe, un variable et une belle participation ! 
* 1 jour de télétravail par semaine * 6 semaines de congés payés par an * Convention collective applicable : Négoce
de l’ameublement 1880 * Mutuelle Alan (60% à la charge de la société) * Swile : carte Tickets restaurant (50% à la 
charge de l\'entreprise) * Participation à l\'abonnement des transports en commun (50% à la charge de 
l\'entreprise) * Pour les minis de la VU team, des places en crèche sont disponibles grâce aux Petits Chaperons 
Rouge ! 🧸 * Dog friendly 🐶 * Localisation : Poste basé à 800m de Paris métro ligne 5 station Hoche ou tram ligne 
3 ou bus 🚇 Tu hésites encore ? Si on matche ensemble ❤️ Tu vas bénéficier de : * De 3 belles terrasses aménagées, 
idéales pour les déjeuners au soleil et les pauses café. * L\'accès à la vente au personnel à prix réduit. * De 
boissons chaudes à volonté : du bon café à grains & des thés /infusions. * Une salle de sport, des flippers, un 
babyfoot et des compétitions de ping-pong. * Des corbeilles de fruits bio. * La journée de solidarité offerte ! *

### Données normalisées

In [86]:
# Afficher 
titre = "structure données normalisées"
sample_file = offres_normalisees[200]

afficher_structure_json(sample_file, titre)
rprint(sample_file)

structure données normalisées
├── id (str)
├── source (str)
├── url (str)
├── titre (str)
├── entreprise (NoneType)
├── description (str)
├── competences 📋 (0 éléments)
├── localisation_ville (str)
├── localisation_dept (str)
├── latitude (float)
├── longitude (float)
├── type_contrat (str)
├── teletravail (NoneType)
├── salaire_min (NoneType)
├── salaire_max (NoneType)
├── salaire_devise (str)
├── experience_min (NoneType)
├── secteur (str)
├── sous_secteur (str)
├── rome_code (str)
├── nb_employes (NoneType)
├── missions 📋 (0 éléments)
├── avantages 📋 (0 éléments)
├── qualification (str)
├── date_publication (str)
└── date_extraction (str)

{
    'id': 'ft_0770443',
    'source': 'francetravail',
    'url': 'https://candidat.francetravail.fr/offres/recherche/detail/0770443',
    'titre': 'Data engineer',
    'entreprise': None,
    'description': "Data engineer F/H \n\nKeolis, leader mondial de la mobilité partagée, facilite le quotidien de 
millions de voyageurs. Nous proposons des solutions de transport en commun sûres, performantes et durables qui 
renforcent l'attractivité des territoires. \n\nEn 2024, le Groupe a réalisé un chiffre d'affaires de 7,7 milliards 
d'euros. Présent dans 13 pays, nous sommes convaincus que nos 70 000 collaborateurs sont la clé de notre 
succès.\n\nContexte\n\nDans le cadre de l'évolution du système d'information de Keolis Rennes Métropole, l'équipe 
Data & Flux se renforce pour accompagner les besoins opérationnels des métiers. Vous intervenez sur l'intégration, 
la transformation et la mise à disposition des données , en lien étroit avec les équipes IT et métiers. Le poste 
s'inscrit dans un environnement collaboratif, avec des enjeux forts de fiabilité, de qualité des données et 
d'industrialisation des flux. \n\nMission\n\nVous serez en charge de :\n\nConstruire, industrialiser et maintenir 
l'intégration des données a) dans Fabric depuis une architecture Data hybride (on-premise et cloud Azure) et b) 
dans Talend pour l'env. on-prem.\n\nTravailler sur des architectures médaillons (Bronze / Silver / 
Gold).\n\nDévelopper et optimiser des traitements de données avancés avec une forte sensibilité à la qualité des 
données.\n\nAider à la conception, réalisation et maintenance des modèles analytiques pour Power BI (modèles en 
étoile).\n\nParticiper à l'architecture, aux bonnes pratiques et à la gouvernance Fabric.\n\nCollaborer avec IT, 
Métier et BI pour cadrer les besoins et proposer des solutions fiables.\n\nRôle de référent Fabric : 
accompagnement, pédagogie, partage de bonnes pratiques.\n\nProfil\n\nVous justifiez d'une première expérience 
réussie en data engineering, vous intervenez sur la conception, le développement et la maintenance de flux de 
données et de traitements analytiques, avec une attention particulière portée à la qualité, la fiabilité et la 
performance des données.Vous maîtrisez les fondamentaux de l'ingénierie data : intégration de données, 
architectures analytiques (notamment médaillon), modélisation dimensionnelle et exploitation des données à des fins
de reporting.\n\nUne connaissance opérationnelle des environnements Microsoft Azure Data (Data Factory, ADLS, Azure
SQL Database) et de Power BI (modèles sémantiques, rapports) constitue un réel atout, tout comme la détention de 
certifications telles que DP-600, DP-700 ou PL-300.\n\nSur le plan humain, vous êtes reconnu(e) pour votre 
excellente communication, votre capacité à échanger aussi bien avec des profils techniques que métiers, et à 
structurer et challenger les besoins pour proposer des solutions pertinentes. Pédagogue et orienté(e) service, vous
savez vulgariser vos sujets, accompagner les utilisateurs et transmettre vos bonnes pratiques. Rigueur, autonomie 
et sens de la qualité complètent votre profil et vous permettent de travailler efficacement dans un environnement 
collaboratif et exigeant. \n\nMaîtrise des services Microsoft Fabric principaux : Lakehouse, Warehouse, Notebooks, 
OneLake, Dataflows Gen2, Pipelines, Power BI\n\nLangages : SQL, PySpark\n\nModélisation dimensionnelle (modèles en 
étoile)\n\nMaîtrise des architectures médaillons (Bronze/Silver/Gold) et de leurs déclinaisons 
opérationnelles\n\nCe que nous avons à offrir :\n\nDate de contrat : CDI dès que possible\n\nRémunération : A 
définir selon profil, grille d'ancienneté avec % versée dès 6 mois, puis à 1 an, puis à 3 ans...\n\nAvantages : 
titres restaurants de 12,10 € (dont 7,26€ pris en charge par l'entreprise), carte mobilité durable, abonnement 
carte Korrigo, Mutuelle d'entreprise Keolis, CSE, \n\nPourquoi nous rejoindre ?\n\nEn intégrant le Groupe Keolis 
vous rej

## Recherche clés dictionnaires

In [87]:
def recup_cles_dico(offre):
    '''
    Récupère les clés et sous-clés d'un dictionnaire de la forme [clé1, clé2__sous-clé1, clé2__sous-clé2, clé3, clé4] 
    Entrée : un dictionnaire
    Sortie : une liste de clés et sous-clés.
    '''
    cles = []
    for key, value in offre.items():   
        # Si la clé lue n'a pas été répertoriée
        if key not in cles :
            
            # Si la valeur de la clé lue est de type dictionnaire
            if isinstance(value, dict):

                # Récupération de toutes les sous-clés ET merge avec la clé actuelle pour traçabilité              
                # Mise en forme {clé : type(clé)} puis ajout à la liste des clés
                cles.extend([{f"{key}__{k}" : type(v)} for k, v in value.items()])
   
            # Sinon, Si la valeur de la clé lue est de type liste
            elif isinstance(value, list):

                #  liste ne contenant PAS de dictionnaire
                if not any(isinstance(element, dict) for element in value):
                    
                    # la clé lue est ajoutée à la liste des clés
                    cles.append({key : type(value)})

                # Si la liste contient au moins un dictionnaire (hypothèse : 1 seul dictionnaire présent dans la liste) 
                else : 
                    # Récupération de toutes les sous-clés ET merge avec la clé actuelle pour traçabilité                   
                    # Mise en forme {clé : type(clé)} puis ajout à la liste des clés
                    cles.extend([{f"{key}__{k}" : type(v)} for k, v in value[0].items()])

            # Sinon, si la clé lue n'est ni de type dictionnaire, ni de type liste
            else:
                # la clé lue est ajoutée à la liste des clés
                cles.append({key : type(value)})
    return cles

#====================================================================================s
def extraire_cles(offres):
    '''
    Récupère les clés et sous-clés d'une liste de dictionnaires de la forme [clé1, clé2__sous-clé1, clé2__sous-clé2, clé3, clé4] 
    Entrée : une liste de dictionnaires
    Sortie : une liste de clés et sous-clés.
    '''
    liste_cles = []
    for offre in offres:
        interim = recup_cles_dico(offre)
        for cle in interim:
            if cle not in liste_cles:
                liste_cles.append(cle)
    return liste_cles

In [88]:
# Clés données brutes
cles_offres_brutes_ft = extraire_cles(offres_brutes_ft)
cles_offres_brutes_wttj = extraire_cles(offres_brutes_wttj)

# Clés données parsees
cles_offres_parsees_ft = extraire_cles(offres_parsees_ft)
cles_offres_parsees_wttj = extraire_cles(offres_parsees_wttj)

# Clés données normalisées
cles_offres_normalisees = extraire_cles(offres_normalisees)

# Nombre de clés
print(f"clés données brutes - France travail - Nombre : {len(cles_offres_brutes_ft)}")
print(f"clés données brutes - Welcome to the Jungle - Nombre : {len(cles_offres_brutes_wttj)}")
print(f"clés données parsees - France travail - Nombre : {len(cles_offres_parsees_ft)}")
print(f"clés données parsees - Welcome to the Jungle - Nombre : {len(cles_offres_parsees_wttj)}")
print(f"clés données normalisées -  Nombre : {len(cles_offres_normalisees)}")

# #Affichage des clés
display(cles_offres_brutes_ft)
display(cles_offres_brutes_wttj)
display(cles_offres_parsees_ft)
display(cles_offres_parsees_wttj)
display(cles_offres_normalisees) 

clés données brutes - France travail - Nombre : 72
clés données brutes - Welcome to the Jungle - Nombre : 104
clés données parsees - France travail - Nombre : 46
clés données parsees - Welcome to the Jungle - Nombre : 40
clés données normalisées -  Nombre : 40


[{'id': str},
 {'intitule': str},
 {'description': str},
 {'dateCreation': str},
 {'dateActualisation': str},
 {'lieuTravail__libelle': str},
 {'romeCode': str},
 {'romeLibelle': str},
 {'appellationlibelle': str},
 {'entreprise__nom': str},
 {'entreprise__entrepriseAdaptee': bool},
 {'typeContrat': str},
 {'typeContratLibelle': str},
 {'natureContrat': str},
 {'experienceExige': str},
 {'experienceLibelle': str},
 {'langues__libelle': str},
 {'langues__exigence': str},
 {'competences__code': str},
 {'competences__libelle': str},
 {'competences__exigence': str},
 {'salaire__libelle': str},
 {'dureeTravailLibelle': str},
 {'dureeTravailLibelleConverti': str},
 {'alternance': bool},
 {'contact__nom': str},
 {'contact__coordonnees1': str},
 {'contact__courriel': str},
 {'nombrePostes': int},
 {'accessibleTH': bool},
 {'deplacementCode': str},
 {'deplacementLibelle': str},
 {'qualificationCode': str},
 {'qualificationLibelle': str},
 {'codeNAF': str},
 {'secteurActivite': str},
 {'secteurA

[{'has_education_level': bool},
 {'contract_duration_maximum': NoneType},
 {'published_at_date': str},
 {'reference': str},
 {'has_salary_yearly_minimum': bool},
 {'salary_currency': str},
 {'new_profession__sub_category_reference': str},
 {'new_profession__sub_category_name': str},
 {'new_profession__category_reference': str},
 {'new_profession__category_name': str},
 {'new_profession__pivot_name': str},
 {'new_profession__pivot_reference': str},
 {'salary_minimum': int},
 {'is_boosted': bool},
 {'benefits': list},
 {'source_stage': str},
 {'has_remote': bool},
 {'contract_type': str},
 {'slug': str},
 {'has_experience_level_minimum': bool},
 {'summary': str},
 {'rank_group_1': int},
 {'organization_score': int},
 {'published_at_timestamp': int},
 {'offices__state': str},
 {'offices__city': str},
 {'offices__country': str},
 {'offices__country_code': str},
 {'offices__district': str},
 {'offices__local_district': str},
 {'offices__local_city': str},
 {'offices__local_state': str},
 {'

[{'id': str},
 {'url': str},
 {'titre': str},
 {'description': str},
 {'entreprise': str},
 {'nb_employes': str},
 {'type_contrat': str},
 {'type_contrat_libelle': str},
 {'alternance': bool},
 {'nombre_postes': int},
 {'temps_travail': str},
 {'salaire_min': int},
 {'salaire_max': int},
 {'salaire_texte': str},
 {'experience_exige': str},
 {'experience_min': int},
 {'qualification': str},
 {'localisation_ville': str},
 {'localisation_dept': NoneType},
 {'commune': NoneType},
 {'latitude': NoneType},
 {'longitude': NoneType},
 {'teletravail': NoneType},
 {'secteur': str},
 {'code_naf': str},
 {'rome_code': str},
 {'rome_libelle': str},
 {'competences': list},
 {'langues': list},
 {'qualites': list},
 {'url_postulation': NoneType},
 {'date_publication': str},
 {'date_actualisation': str},
 {'date_extraction': str},
 {'source': str},
 {'entreprise': NoneType},
 {'nb_employes': NoneType},
 {'salaire_min': NoneType},
 {'salaire_max': NoneType},
 {'salaire_texte': NoneType},
 {'localisation

[{'id': str},
 {'slug': str},
 {'url': str},
 {'titre': str},
 {'description': str},
 {'missions': list},
 {'type_contrat': str},
 {'teletravail': str},
 {'salaire_min': int},
 {'salaire_max': int},
 {'salaire_devise': str},
 {'experience_min': int},
 {'localisation_ville': str},
 {'localisation_region': str},
 {'pays': str},
 {'latitude': float},
 {'longitude': float},
 {'entreprise': str},
 {'entreprise_slug': str},
 {'nb_employes': int},
 {'entreprise_desc': str},
 {'secteur': str},
 {'sous_secteur': str},
 {'avantages': list},
 {'metier': str},
 {'categorie_metier': str},
 {'date_publication': str},
 {'date_extraction': str},
 {'source': str},
 {'salaire_max': NoneType},
 {'salaire_min': NoneType},
 {'localisation_region': NoneType},
 {'salaire_devise': NoneType},
 {'experience_min': NoneType},
 {'experience_min': float},
 {'latitude': NoneType},
 {'longitude': NoneType},
 {'nb_employes': NoneType},
 {'secteur': NoneType},
 {'sous_secteur': NoneType}]

[{'id': str},
 {'source': str},
 {'url': str},
 {'titre': str},
 {'entreprise': str},
 {'description': str},
 {'competences': list},
 {'localisation_ville': str},
 {'localisation_dept': NoneType},
 {'latitude': NoneType},
 {'longitude': NoneType},
 {'type_contrat': str},
 {'teletravail': NoneType},
 {'salaire_min': int},
 {'salaire_max': int},
 {'salaire_devise': str},
 {'experience_min': int},
 {'secteur': str},
 {'sous_secteur': str},
 {'rome_code': str},
 {'nb_employes': str},
 {'missions': list},
 {'avantages': list},
 {'qualification': str},
 {'date_publication': str},
 {'date_extraction': str},
 {'entreprise': NoneType},
 {'localisation_dept': str},
 {'latitude': float},
 {'longitude': float},
 {'salaire_min': NoneType},
 {'salaire_max': NoneType},
 {'nb_employes': NoneType},
 {'experience_min': NoneType},
 {'teletravail': str},
 {'nb_employes': int},
 {'salaire_devise': NoneType},
 {'experience_min': float},
 {'secteur': NoneType},
 {'sous_secteur': NoneType}]